In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor


In [3]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [4]:

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")


In [5]:
from openai import OpenAI
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

COMMIT = "8c1834d"
load_dotenv()

    

True

In [6]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [7]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
rag = RAGBase(index=index, llm_client=client)

In [8]:
class RAGTraced(RAGBase):

    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as llm:
            response = super().llm(prompt)
            llm.set_attribute("input_tokens",response.usage.input_tokens)
            llm.set_attribute("output_tokens",response.usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [9]:
rag_traced = RAGTraced(
    rag.index,
    rag.llm_client,
    rag.model
)

In [12]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_traced.rag(query)

print(answer)

The agentic loop keeps calling the model by using a `while True` loop and a simple stop flag:

1. Call the model with the current `messages`.
2. Check the model’s output:
   - If it includes a `function_call`, run the tool, append the tool result to `messages`, and set `has_function_calls = True`.
   - If it includes only a final `message`, print it.
3. If `has_function_calls` is `False`, break out of the loop.

So the loop continues as long as the model asks for tools, and it stops when the model returns a response with no function calls.

In code, the key part is:

```python
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

if has_function_calls == False:
    break
```

In short: **the model decides whether more tool calls are needed, and your code keeps looping until it doesn’t request any more.**


In [16]:
import pandas as pd
conn = sqlite3.connect("traces.db")

df = pd.read_sql_query("SELECT * FROM spans", conn)
df.columns = ["span_name", "start_time", "end_time", "input_tokens", "output_tokens","cost"]

llm = df[df["span_name"] == "llm"]
print(llm["input_tokens"])
print("Unique values:", llm["input_tokens"].unique())

1     7070.0
4     7070.0
7     7070.0
10    7070.0
Name: input_tokens, dtype: float64
Unique values: [7070.]
